   
Databricks notebook source
SENSE 프로젝트 — Silver Layer: FX raw → silver
담당: 1조 (정형 데이터)

목적:
  ADLS Gen2 raw/fx/ 에서 1년치 USD/KRW 환율 CSV 데이터를 읽어
  master_calendar 기반 시계열 정제 후 curated/fx/ 에 저장

처리 대상:
  USD/KRW 환율 (source: 1년치 CSV, 2025-04-06 ~ 2026-04-06)

[파생 컬럼]
  usd_krw           : 일별 환율
  usd_krw_change    : 전일 대비 환율 변화량, 첫 행 0
  usd_krw_pct       : 전일 대비 환율 변화율 (%), 첫 행 0
  risk_off_flag     : 환율 급등 신호 (전일 대비 +1% 이상 상승 시 1)

# 0. 스토리지 계정 설정 및 ADLS OAuth 인증


In [0]:
# ============================================================
# 스토리지 계정 및 경로 상수 정의
# ============================================================
STORAGE_ACCOUNT  = "3dtteam1adls"
SECRET_SCOPE     = "sense-kv"
BRONZE_CONTAINER    = "raw"
SILVER_CONTAINER = "curated"

BASE_PATH_BRONZE    = f"abfss://{BRONZE_CONTAINER}@{STORAGE_ACCOUNT}.dfs.core.windows.net"
BASE_PATH_SILVER = f"abfss://{SILVER_CONTAINER}@{STORAGE_ACCOUNT}.dfs.core.windows.net"

# master_calendar 경로
CALENDAR_PATH = f"{BASE_PATH_SILVER}/master_calendar.parquet"

# ============================================================
# ADLS Gen2 OAuth 인증 (Service Principal)
# ============================================================
spark.conf.set(
    f"fs.azure.account.auth.type.{STORAGE_ACCOUNT}.dfs.core.windows.net",
    "OAuth"
)
spark.conf.set(
    f"fs.azure.account.oauth.provider.type.{STORAGE_ACCOUNT}.dfs.core.windows.net",
    "org.apache.hadoop.fs.azurebfs.oauth2.ClientCredsTokenProvider"
)
spark.conf.set(
    f"fs.azure.account.oauth2.client.id.{STORAGE_ACCOUNT}.dfs.core.windows.net",
    dbutils.secrets.get(scope=SECRET_SCOPE, key="adls-client-id")
)
spark.conf.set(
    f"fs.azure.account.oauth2.client.secret.{STORAGE_ACCOUNT}.dfs.core.windows.net",
    dbutils.secrets.get(scope=SECRET_SCOPE, key="adls-client-secret")
)
spark.conf.set(
    f"fs.azure.account.oauth2.client.endpoint.{STORAGE_ACCOUNT}.dfs.core.windows.net",
    f"https://login.microsoftonline.com/{dbutils.secrets.get(scope=SECRET_SCOPE, key='adls-tenant-id')}/oauth2/token"
)

print(f"✅ ADLS OAuth 인증 설정 완료: {STORAGE_ACCOUNT}")


# 1. master_calendar 로드

FX 데이터는 하루에도 여러 번 수집되기 때문에
캘린더의 **한국 영업일** 기준으로 유효 거래일을 필터합니다.

> 환율은 한국 시장이 열린 날이 기준입니다.
> 한국 공휴일(예: 추석)에는 환전 거래가 없으므로 해당 날짜는 제외합니다.


In [0]:
import pyspark.sql.functions as F
from pyspark.sql.functions import to_date, col

# master_calendar 로드
calendar_df = (
    spark.read
    .parquet(CALENDAR_PATH)
    .withColumn("기준일자", to_date(col("기준일자"), "yyyy-MM-dd"))
    .select("기준일자", "주말여부", "한국_휴장일_여부", "미국_휴장일_여부")
)

# 한국 실제 영업일
kr_biz_days = (
    calendar_df
    .filter(col("한국_휴장일_여부") == False)
    .select(col("기준일자").alias("kr_biz_date"))
)

print(f"✅ master_calendar 로드 완료")
print(f"   전체 기간    : {calendar_df.count()}일")
print(f"   한국 영업일  : {kr_biz_days.count()}일")
display(calendar_df.limit(10))


   
# 2. FX raw 데이터 로드

raw 경로:
```
raw/fx/usd_krw/usd_krw_250406-260406.csv
```

> 1년치 USD/KRW 환율 CSV 파일 (2025-04-06 ~ 2026-04-06)
> **일별 1행** 구조 — 시간별 집계 불필요

실제 컬럼 구조:
| 컬럼 | 설명 |
|---|---|
| `date` | 기준일자 (DateType) |
| `base` | 기준 통화 (USD) |
| `quote` | 환산 통화 (KRW) |
| `rate` | 환율 (USD 1 = KRW rate) |

> 주말 포함 365일 → master_calendar로 한국 영업일 필터 필요

In [0]:
fx_raw_경로 = f"{BASE_PATH_BRONZE}/fx/usd_krw/usd_krw_250406-260406.csv"

fx_raw_df = (
    spark.read
    .format("csv")
    .option("header", "true")
    .option("inferSchema", "true")
    .option("encoding", "utf-8")
    .load(fx_raw_경로)
)

print(f"✅ FX raw 로드 완료: {fx_raw_df.count()}행")
print(f"   컬럼: {fx_raw_df.columns}")
fx_raw_df.printSchema()
display(fx_raw_df.limit(10))

   
# 3. 스키마 정리

- CSV에 `date` (DateType)가 이미 존재 → 별도 변환 불필요
- `rate` → `usd_krw` 리네임
- `base`, `quote` 컬럼 제거 (모두 USD/KRW 고정)
- null 제거: `date`, `usd_krw`

In [0]:
from pyspark.sql.functions import to_date, year, month, col

fx_clean_df = (
    fx_raw_df
    .select(
        col("date"),
        col("rate").alias("usd_krw")
    )
    .filter(col("date").isNotNull())
    .filter(col("usd_krw").isNotNull())
    .orderBy("date")
)

print(f"✅ 스키마 정리 완료: {fx_clean_df.count()}행")
display(fx_clean_df.limit(10))

# 5. 한국 영업일 필터 (master_calendar 기반)

집계된 일별 환율에서 **한국 실제 영업일만 남깁니다.**

> 주말이나 한국 공휴일(추석, 설날 등)에도
> openexchangerates는 계속 데이터를 수집합니다.
> 하지만 한국 외환시장이 닫힌 날의 환율은
> Gold JOIN 시 불필요하므로 제거합니다.


In [0]:
fx_filtered_df = (
    fx_clean_df
    .join(
        kr_biz_days.withColumnRenamed("kr_biz_date", "date"),
        on="date",
        how="inner"   # 한국 영업일만 남김, 공휴일은 자동 제거
    )
)

before = fx_clean_df.count()
after  = fx_filtered_df.count()

print(f"✅ 한국 영업일 필터 완료")
print(f"   필터 전: {before}일  →  필터 후: {after}일")
print(f"   제거된 비영업일: {before - after}일 (주말 + 공휴일)")
display(fx_filtered_df.orderBy("date").limit(10))

# 6. 파생 컬럼 생성

| 컬럼명 | 계산 방법 | 의미 |
|---|---|---|
| `usd_krw_change` | 오늘 환율 - 어제 환율 | 전일 대비 환율 변화량 (원) |
| `usd_krw_pct` | 변화량 / 어제 환율 × 100 | 전일 대비 환율 변화율 (%) |
| `risk_off_flag` | usd_krw_pct ≥ +1.0% 이면 1 | 달러 급등 신호 |

### 쉬운 설명

**risk_off_flag (위험 회피 신호)**
> 하루 만에 환율이 1% 이상 오른다는 건 외국인 투자자들이
> 한국 주식을 팔고 달러를 사서 빠져나가고 있다는 신호입니다.
> 이 플래그가 1이 되는 날은 반도체 주가 하락과 강하게 연동됩니다.


In [0]:
from pyspark.sql import Window
from pyspark.sql.functions import lag, when, round as spark_round

w_date = Window.orderBy("date")

fx_featured_df = (
    fx_filtered_df

    # 전일 대비 변화량
    .withColumn(
        "usd_krw_change",
        F.round(col("usd_krw") - lag("usd_krw", 1).over(w_date), 4)
    )

    # 전일 대비 변화율 (%)
    .withColumn(
        "usd_krw_pct",
        F.round(
            (col("usd_krw") - lag("usd_krw", 1).over(w_date))
            / lag("usd_krw", 1).over(w_date) * 100,
            4
        )
    )

    # 달러 급등 신호 (1% 이상 상승)
    .withColumn(
        "risk_off_flag",
        when(col("usd_krw_pct") >= 1.0, 1).otherwise(0)
    )

    # 첫 행 null → 0 채우기 (lag 초기값 부재)
    .na.fill(0.0, subset=["usd_krw_change", "usd_krw_pct"])

    # 파티션 컬럼
    .withColumn("year",  F.year("date"))
    .withColumn("month", F.month("date"))
)

print("✅ 파생 컬럼 생성 완료")
display(
    fx_featured_df
    .select("date", "usd_krw", "usd_krw_change", "usd_krw_pct", "risk_off_flag")
    .orderBy("date")
    .limit(15)
)

# 7. 데이터 검증


In [0]:
# ── 날짜 범위 확인 ──────────────────────────────────────────────────
date_range = fx_featured_df.agg(
    F.min("date").alias("start_date"),
    F.max("date").alias("end_date"),
    F.count("date").alias("영업일수")
).collect()[0]

print("=== 📅 날짜 범위 ===")
print(f"  시작: {date_range['start_date']}")
print(f"  종료: {date_range['end_date']}")
print(f"  한국 영업일수: {date_range['영업일수']}일")

# ── null 비율 확인 ───────────────────────────────────────────────────
print("\n=== 🔍 컬럼별 null 비율 ===")
total = fx_featured_df.count()
for c in ["usd_krw", "usd_krw_change", "usd_krw_pct", "risk_off_flag"]:
    null_cnt = fx_featured_df.filter(col(c).isNull()).count()
    status = "✅" if null_cnt == 0 else "⚠️"
    print(f"  {status} {c}: null {null_cnt}건 ({null_cnt/total*100:.1f}%)")
print("  (usd_krw_change, usd_krw_pct 첫 행은 0으로 대체 완료)")

# ── risk_off_flag 발생 날짜 확인 ─────────────────────────────────────
print("\n=== 🚨 risk_off_flag 발생일 (달러 급등 +1% 이상) ===")
risk_days = fx_featured_df.filter(col("risk_off_flag") == 1)
print(f"  총 {risk_days.count()}건")
display(
    risk_days
    .select("date", "usd_krw", "usd_krw_change", "usd_krw_pct")
    .orderBy("date")
)

# 8. Silver 저장

- 포맷: **Parquet** (기존 노트북 방식 유지)
- 파티션: `year` / `month`
- 모드: `overwrite`

> ⚠️ overwrite 사용 이유:
> `usd_krw_change`, `usd_krw_pct` 파생 컬럼이 전체 시계열 참조
> → 매번 전체 재처리


In [0]:
SILVER_PATH = f"{BASE_PATH_SILVER}/fx"

(
    fx_featured_df
    .write
    .mode("overwrite")
    .partitionBy("year", "month")
    .parquet(SILVER_PATH)
)

saved_files = dbutils.fs.ls(SILVER_PATH)
print(f"✅ Silver 저장 완료")
print(f"   경로     : {SILVER_PATH}")
print(f"   파티션 수: {len(saved_files)}개")
for f in saved_files:
    print(f"   - {f.name}")


# 9. 저장 확인 (sanity check)


In [0]:
df_check = spark.read.parquet(SILVER_PATH)

print(f"✅ silver 읽기 확인 — 행수: {df_check.count():,}")
print(f"   컬럼: {df_check.columns}")
display(df_check.orderBy("date").limit(10))


   
## curated/fx Silver 컬럼 요약 (243행 × 7컬럼)

| # | 컬럼 | 타입 | 설명 |
|---|---|---|---|
| 1 | `date` | DateType | 기준일자 (한국 영업일) |
| 2 | `usd_krw` | DoubleType | USD/KRW 일별 환율 |
| 3 | `usd_krw_change` | DoubleType | 전일 대비 환율 변화량 (원), 첫 행 0 |
| 4 | `usd_krw_pct` | DoubleType | 전일 대비 환율 변화율 (%), 첫 행 0 |
| 5 | `risk_off_flag` | IntegerType | 달러 급등 신호 (≥1% 상승 시 1) |
| 6 | `year` | IntegerType | 파티션 컬럼 (연도) |
| 7 | `month` | IntegerType | 파티션 컬럼 (월) |

**기간**: 2025-04-07 ~ 2026-04-03 · null 0건 · risk_off 2건